In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
!pip install faiss-cpu langchain langchain-community langchain-core langchain-huggingface pypdf sentence-transformers transformers torch

In [ ]:
!pip install unsloth

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from unsloth import FastLanguageModel
import torch

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Malakkkkk/egyptian-complaint-model",  
    max_seq_length=2048,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

In [ ]:
import re, json

def extract_json_block(text):
    pattern = r'```json\s*(.*?)\s*```'
    matches = re.findall(pattern, text, re.DOTALL)
    if matches:
        return matches[-1]
    start, end = text.find("{"), text.rfind("}")
    if start != -1 and end != -1:
        return text[start:end + 1]
    return text



In [ ]:
from langchain_classic.output_parsers import ResponseSchema, StructuredOutputParser


category_schemas = [
    ResponseSchema(name="category", description="delivery, telecom, or ride_hailing"),
    ResponseSchema(name="subcategory", description="specific issue type"),
]
category_parser = StructuredOutputParser.from_response_schemas(category_schemas)

decision_schemas = [
    ResponseSchema(name="severity", description="low, medium, or high"),
    ResponseSchema(name="refund_eligible", description="yes, no, or partial"),  
    ResponseSchema(name="suggested_reply", description="a short Arabic reply to the customer"),
]
decision_parser = StructuredOutputParser.from_response_schemas(decision_schemas)


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

policies = [
    # delivery
    {"category": "delivery", "subcategory": "cold_food",
     "text": "لو الأكل وصل بارد بسبب تأخير التوصيل، العميل يستحق كوبون خصم 15% على الطلب الجاي."},
    {"category": "delivery", "subcategory": "damaged_package",
     "text": "لو المنتج وصل تالف أو الطلب مكسور، العميل يستحق استرداد كامل أو استبدال المنتج مجانًا."},
    {"category": "delivery", "subcategory": "delivery_delay",
     "text": "التأخير في التوصيل لأكتر من ساعة عن الوقت المتوقع يستحق تعويض بقيمة 20% من قيمة الطلب."},
    {"category": "delivery", "subcategory": "double_charge",
     "text": "لو العميل اتخصم منه مرتين على نفس الطلب، يتم استرداد المبلغ الزيادة بالكامل خلال 3 أيام عمل."},
    {"category": "delivery", "subcategory": "expired_product",
     "text": "لو المنتج المستلم منتهي الصلاحية، العميل يستحق استرداد كامل فورًا وإبلاغ المطعم/المتجر."},
    {"category": "delivery", "subcategory": "missing_items",
     "text": "لو جزء من الطلب ناقص، العميل يستحق استرداد قيمة الأصناف الناقصة بالكامل."},
    {"category": "delivery", "subcategory": "rude_courier",
     "text": "شكاوى سلوك المندوب يتم مراجعتها فوريًا، والعميل يستحق كوبون خصم 10% على الطلب الجاي."},
    {"category": "delivery", "subcategory": "wrong_order",
     "text": "لو الطلب اللي وصل غلط تمامًا، العميل يستحق استرداد كامل أو إعادة إرسال الطلب الصحيح مجانًا."},

    # ride_hailing
    {"category": "ride_hailing", "subcategory": "charged_no_ride",
     "text": "لو العميل اتخصم منه فلوس رحلة ملغاة أو ملغيتش، يستحق استرداد كامل للمبلغ خلال 24 ساعة."},
    {"category": "ride_hailing", "subcategory": "driver_cancelled",
     "text": "لو السائق لغى الرحلة بعد قبولها بدون سبب واضح، العميل يستحق كوبون خصم 10% على الرحلة الجاية."},
    {"category": "ride_hailing", "subcategory": "long_wait",
     "text": "لو وقت الانتظار للسائق تجاوز 15 دقيقة عن الوقت المتوقع، العميل يستحق كوبون خصم 10%."},
    {"category": "ride_hailing", "subcategory": "lost_item",
     "text": "الأغراض المفقودة في الرحلة يتم التواصل مع السائق للاسترجاع، ولو لم يتم الاسترجاع خلال 48 ساعة يستحق العميل تعويض."},
    {"category": "ride_hailing", "subcategory": "overcharging_fare",
     "text": "لو السعر النهائي أعلى من المتوقع بدون سبب واضح، العميل يستحق استرداد الفرق بالكامل."},
    {"category": "ride_hailing", "subcategory": "rude_unsafe_driver",
     "text": "شكاوى سلوك السائق يتم مراجعتها فوريًا، والعميل يستحق كوبون خصم 25% على الرحلة الجاية."},
    {"category": "ride_hailing", "subcategory": "wrong_route",
     "text": "لو السائق مشى في طريق غير المتوقع وزود السعر، العميل يستحق استرداد فرق السعر الناتج عن الطريق الزيادة."},

    # telecom
    {"category": "telecom", "subcategory": "bad_customer_service",
     "text": "شكاوى خدمة العملاء يتم تصعيدها للمشرف المباشر، والعميل يستحق اعتذار رسمي ومتابعة خلال 48 ساعة."},
    {"category": "telecom", "subcategory": "billing_overcharge",
     "text": "الخصومات الغلط في الفاتورة يتم استردادها بالكامل خلال 3 أيام عمل."},
    {"category": "telecom", "subcategory": "data_bundle_not_applied",
     "text": "لو الباقة المدفوعة متتفعلش خلال ساعة من الدفع، العميل يستحق استرداد كامل أو تفعيل فوري للباقة."},
    {"category": "telecom", "subcategory": "network_outage",
     "text": "انقطاع الخدمة لأكتر من ساعتين متواصلة، أو تكرار الانقطاع 3 مرات أو أكتر في الأسبوع، يستحق استرداد جزئي بقيمة يوم واحد من قيمة الاشتراك الشهري."},
    {"category": "telecom", "subcategory": "roaming_charges",
     "text": "رسوم الرومنج الغير متوقعة يتم مراجعتها، ولو ثبت عدم تفعيل العميل للخدمة يستحق استرداد كامل."},
    {"category": "telecom", "subcategory": "sim_issue",
     "text": "مشاكل الشريحة (تفعيل، تلف، فقدان الشبكة) يتم حلها مجانًا خلال 24 ساعة بدون أي رسوم إضافية."},
    {"category": "telecom", "subcategory": "slow_internet",
     "text": "بطء الإنترنت المستمر لأكتر من 48 ساعة بعد الإبلاغ يستحق تعويض بقيمة 10% من الفاتورة الشهرية."},
]

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
texts = [p["text"] for p in policies]
metadatas = [{"category": p["category"], "subcategory": p["subcategory"]} for p in policies]
policy_vectordb = FAISS.from_texts(texts, embeddings, metadatas=metadatas)

def retrieve_policy(complaint_text, category=None, subcategory=None, k=1):
    if category and subcategory:
        results = policy_vectordb.similarity_search(
            complaint_text, k=k,
            filter={"category": category, "subcategory": subcategory}
        )
        if results:
            return results[0].page_content

    if category:
        results = policy_vectordb.similarity_search(complaint_text, k=k, filter={"category": category})
        if results:
            return results[0].page_content

    results = policy_vectordb.similarity_search(complaint_text, k=k)
    return results[0].page_content if results else ""

In [ ]:
def run_model(prompt):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    ).to(model.device)
    output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    raw = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return extract_json_block(raw)


def classify_category(complaint_text):
    prompt = "صنّف الشكوى دي وحدد نوعها ونوعها الفرعي بس، من غير ما تحدد الخطورة أو الاسترداد.\n" + complaint_text
    raw = run_model(prompt)
    return category_parser.parse(raw)


def generate_decision(complaint_text, policy_snippet):

    prompt = (
        "دي سياسة الشركة الرسمية اللي بتنطبق على الشكوى دي:\n"
        f"{policy_snippet}\n\n"
        "اعتمد فقط على السياسة دي عشان تحدد درجة الخطورة وهل العميل يستحق استرداد ولا لأ ونوعه، "
        "واقترح رد للعميل يعكس السياسة دي بالظبط.\n\n"
        "الشكوى:\n" + complaint_text
    )
    raw = run_model(prompt)
    return decision_parser.parse(raw)




In [ ]:
REFUND_RANK = {"no": 0, "partial": 1, "yes": 2}

MIN_ENTITLEMENT = {
    ("delivery", "cold_food"): "partial",
    ("delivery", "damaged_package"): "yes",
    ("delivery", "delivery_delay"): "partial",
    ("delivery", "double_charge"): "yes",
    ("delivery", "expired_product"): "yes",
    ("delivery", "missing_items"): "partial",
    ("delivery", "rude_courier"): "partial",
    ("delivery", "wrong_order"): "yes",
    ("ride_hailing", "charged_no_ride"): "yes",
    ("ride_hailing", "driver_cancelled"): "partial",
    ("ride_hailing", "long_wait"): "partial",
    ("ride_hailing", "lost_item"): "partial",
    ("ride_hailing", "overcharging_fare"): "yes",
    ("ride_hailing", "rude_unsafe_driver"): "partial",
    ("ride_hailing", "wrong_route"): "yes",
    ("telecom", "billing_overcharge"): "yes",
    ("telecom", "data_bundle_not_applied"): "yes",
    ("telecom", "network_outage"): "partial",
    ("telecom", "roaming_charges"): "partial",
    ("telecom", "slow_internet"): "partial",
}


OVERRIDE_REPLY_TEMPLATES = {
    "yes": "بعد المراجعة، تم اعتماد استرداد كامل لقيمة الطلب طبقًا لسياسة الشركة.",
    "partial": "بعد المراجعة، تم اعتماد تعويض جزئي طبقًا لسياسة الشركة.",
}

def enforce_policy_minimum(category, subcategory, model_refund_eligible):
    minimum = MIN_ENTITLEMENT.get((category, subcategory))
    if minimum is None:
        return model_refund_eligible, False  

    model_rank = REFUND_RANK.get(model_refund_eligible, 0)
    min_rank = REFUND_RANK.get(minimum, 0)

    if model_rank < min_rank:
        return minimum, True
    return model_refund_eligible, False

In [ ]:
CORRECTION_NOTES = {
    "yes": " تحديث: بعد المراجعة، تم اعتماد استرداد كامل طبقًا لسياسة الشركة.",
    "partial": " تحديث: بعد المراجعة، تم اعتماد تعويض جزئي طبقًا لسياسة الشركة.",
}

def process_complaint(complaint_text):
    try:
        category_result = classify_category(complaint_text)
    except Exception as e:
        return {"error": f"Category classification failed: {e}"}

    category = category_result.get("category")
    subcategory = category_result.get("subcategory")
    policy_snippet = retrieve_policy(complaint_text, category=category, subcategory=subcategory)

    try:
        decision = generate_decision(complaint_text, policy_snippet)
    except Exception as e:
        return {
            "error": f"Decision generation failed: {e}",
            "category": category, "subcategory": subcategory,
            "policy_used": policy_snippet,
        }

    model_refund = decision.get("refund_eligible")
    final_refund, was_overridden = enforce_policy_minimum(category, subcategory, model_refund)

    reply = decision.get("suggested_reply", "")
    if was_overridden:
        reply = reply + CORRECTION_NOTES.get(final_refund, "")

    return {
        "category": category,
        "subcategory": subcategory,
        "severity": decision.get("severity"),
        "refund_eligible": final_refund,
        "policy_override_applied": was_overridden,
        "suggested_reply": reply,
        "policy_used": policy_snippet,
    }

In [ ]:
result = process_complaint("الاكل جه بارد بجد")
print(result)

In [ ]:
pip install fastapi uvicorn pyngrok accelerate -q

In [ ]:
NGROK_TOKEN = "3GltTYylsrwnFTT5LU39i71dgbs_4Sfdke6SDzYS4fNiyR5Wy"
API_KEY = "secret345"

In [ ]:
from fastapi import FastAPI, Request, HTTPException
import uvicorn, threading, time, socket
from pyngrok import ngrok, conf
from kaggle_secrets import UserSecretsClient

app = FastAPI()

@app.post("/process_complaint")
async def process(req: Request):
    if req.headers.get("authorization") != f"Bearer {API_KEY}":
        raise HTTPException(status_code=401, detail="Unauthorized")
    data = await req.json()
    complaint_text = data.get("complaint", "")
    if not complaint_text:
        raise HTTPException(status_code=400, detail="Missing 'complaint' field")
    return process_complaint(complaint_text)

def free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

port = free_port()
user_secrets = UserSecretsClient()
conf.get_default().auth_token = user_secrets.get_secret("NGROK_TOKEN")
public_url = ngrok.connect(port).public_url
print("Your public URL:", public_url)

def run(): uvicorn.run(app, host="0.0.0.0", port=port)
threading.Thread(target=run, daemon=True).start()
time.sleep(1)

In [ ]:
import requests

res = requests.post(
    f"http://localhost:{port}/process_complaint",
    headers={"Authorization": f"Bearer {API_KEY}"},
    json={"complaint": "النت بيقطع بقاله اسبوع"},
    timeout=120,
)
print(res.status_code, res.text)